# Games Backtest (2024 default)

Backtest variant of `games_today_tomorrow.ipynb`:

- Defaults to **completed 2024 games** from `data/games_2024.parquet`.
- Keeps V6-style scoring logic, but changes data selection upstream.
- If Kalshi historical implied columns are missing, scoring still runs and Kalshi-based metrics are omitted (set to NaN).

Run top-to-bottom after refreshing Parquet files.

In [10]:
# Backtest parameters (edit these)
SEASON = 2024
START_DATE = "2024-03-28"  # inclusive, YYYY-MM-DD
END_DATE = "2024-11-11"    # inclusive; None = today

FINAL_STATES = {"Final", "Game Over", "Completed Early"}

print({
    "SEASON": SEASON,
    "START_DATE": START_DATE,
    "END_DATE": END_DATE or "today",
    "FINAL_STATES": sorted(FINAL_STATES),
})

{'SEASON': 2024, 'START_DATE': '2024-03-28', 'END_DATE': '2024-11-11', 'FINAL_STATES': ['Completed Early', 'Final', 'Game Over']}


In [11]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

cwd = Path.cwd()
DATA = cwd / "data" if (cwd / "data").is_dir() else cwd.parent / "data"
PARQUET = DATA / f"games_{SEASON}.parquet"

if not PARQUET.is_file():
    raise FileNotFoundError(f"Missing {PARQUET}. Run: python -m app.main live --season {SEASON}")

df = pd.read_parquet(PARQUET)
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce").dt.normalize()
df["detailed_state"] = df["detailed_state"].astype(str)

df_bt = df[df["detailed_state"].isin(FINAL_STATES)].copy()

if "home_win" in df_bt.columns:
    # ensure final training label is present for backtest
    df_bt = df_bt[df_bt["home_win"].notna()].copy()

start_ts = pd.Timestamp(START_DATE).normalize()
end_ts = pd.Timestamp.today().normalize() if END_DATE is None else pd.Timestamp(END_DATE).normalize()
if end_ts < start_ts:
    raise ValueError(f"END_DATE ({end_ts.date()}) must be >= START_DATE ({start_ts.date()})")

sub = df_bt[df_bt["game_date"].between(start_ts, end_ts)].copy()

print("Using:", PARQUET.resolve())
print(f"Rows total={len(df)}  completed={len(df_bt)}")
if len(df_bt):
    print(f"Date range completed: {df_bt['game_date'].min().date()} .. {df_bt['game_date'].max().date()}")
print(f"Backtest window: {start_ts.date()} .. {end_ts.date()}  rows={len(sub)}")


Using: C:\Users\Austin\baseball\mlb-model\data\games_2024.parquet
Rows total=3093  completed=2915
Date range completed: 2024-03-01 .. 2024-10-30
Backtest window: 2024-03-28 .. 2024-11-11  rows=2601


In [12]:
# Core thresholds from your mismatch workflow
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
MIN_CORE_MATCHES = 2

USE_PLATOON_EXTRA = True
PLATOON_MIN = 0.0
USE_PEN_EXTRA = True
PEN_ROLL_MIN_GAP = 0.0

s = sub.copy()
req = {"sp_kbb_diff", "sp_xfip_diff", "offense_diff"}
missing = req - set(s.columns)
if missing:
    raise ValueError(f"Missing columns for mismatch filter: {missing}")

# Core directional checks
home_core_kbb = s["sp_kbb_diff"] > SP_KBB_ABS
home_core_xfip = s["sp_xfip_diff"] < -SP_XFIP_ABS
home_core_off = s["offense_diff"] > OFFENSE_ABS
home_core_n = home_core_kbb.astype(int) + home_core_xfip.astype(int) + home_core_off.astype(int)

away_core_kbb = s["sp_kbb_diff"] < -SP_KBB_ABS
away_core_xfip = s["sp_xfip_diff"] > SP_XFIP_ABS
away_core_off = s["offense_diff"] < -OFFENSE_ABS
away_core_n = away_core_kbb.astype(int) + away_core_xfip.astype(int) + away_core_off.astype(int)

home_base = home_core_n >= MIN_CORE_MATCHES
away_base = away_core_n >= MIN_CORE_MATCHES

home_ext = home_base.copy()
away_ext = away_base.copy()

if USE_PLATOON_EXTRA and "offense_platoon_diff" in s.columns:
    po = s["offense_platoon_diff"].fillna(0)
    home_ext = home_ext & (po > PLATOON_MIN)
    away_ext = away_ext & (po < -PLATOON_MIN)

if USE_PEN_EXTRA and {"home_pen_roll14_fip", "away_pen_roll14_fip"}.issubset(s.columns):
    hr = s["home_pen_roll14_fip"]
    ar = s["away_pen_roll14_fip"]
    g = PEN_ROLL_MIN_GAP
    pen_ok_home = (hr + g < ar) | hr.isna() | ar.isna()
    pen_ok_away = (ar + g < hr) | hr.isna() | ar.isna()
    home_ext = home_ext & pen_ok_home
    away_ext = away_ext & pen_ok_away

favorites = s.copy().sort_values(["game_date", "home_team_name"]).reset_index(drop=True)
favorites["home_core_matches"] = home_core_n.astype(int)
favorites["away_core_matches"] = away_core_n.astype(int)
favorites["home_base_match"] = home_base
favorites["away_base_match"] = away_base
favorites["home_with_extras"] = home_ext
favorites["away_with_extras"] = away_ext

_tie_home = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + (-favorites["sp_xfip_diff"]).abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
_tie_away = (
    favorites["sp_kbb_diff"].abs() / SP_KBB_ABS
    + favorites["sp_xfip_diff"].abs() / SP_XFIP_ABS
    + favorites["offense_diff"].abs() / OFFENSE_ABS
)
prefer_home = (favorites["home_core_matches"] > favorites["away_core_matches"]) | (
    (favorites["home_core_matches"] == favorites["away_core_matches"]) & (_tie_home >= _tie_away)
)

favorites["_mismatch_side"] = prefer_home.map({True: "home_favored", False: "away_favored"})
favorites["core_matches"] = favorites[["home_core_matches", "away_core_matches"]].max(axis=1)
favorites["favored_team"] = np.where(
    favorites["_mismatch_side"].eq("home_favored"),
    favorites["home_team_name"],
    favorites["away_team_name"],
)

print("Favorites frame rows:", len(favorites))
display(favorites.tail(25))

Favorites frame rows: 2601


,game_pk,game_date,detailed_state,home_team_name,away_team_name,home_team_id,away_team_id,home_score,away_score,home_probable_pitcher,away_probable_pitcher,home_probable_pitcher_id,away_probable_pitcher_id,home_sp_throws,away_sp_throws,home_wrc_plus,away_wrc_plus,home_sp_kbb,away_sp_kbb,home_sp_xfip,away_sp_xfip,home_win,sp_kbb_diff,sp_xfip_diff,offense_diff,home_offense_platoon,away_offense_platoon,offense_platoon_diff,home_sp_kbb_roll14,away_sp_kbb_roll14,home_sp_xfip_roll14,away_sp_xfip_roll14,home_pen_season_fip,away_pen_season_fip,home_pen_season_era,away_pen_season_era,home_pen_roll14_fip,away_pen_roll14_fip,home_pen_roll14_era,away_pen_roll14_era,home_pen_roll14_minus_season_fip,away_pen_roll14_minus_season_fip,kalshi_home_implied,kalshi_away_implied,edge_vs_model,stats_season,home_core_matches,away_core_matches,home_base_match,away_base_match,home_with_extras,away_with_extras,_mismatch_side,core_matches,favored_team
2576,775321,2024-10-08,Final,San Diego Padres,Los Angeles Dodgers,135,119,6.0,5.0,Michael King,Walker Buehler,650633,621111,R,R,104.665885,109.871043,19.008264,10.465116,3.266987,5.476106,1.0,8.543148,-2.209120,-5.205158,107.348602,109.034706,-1.686104,5.000000,0.000000,3.700000,3.30,3.289800,3.636504,3.573273,3.200938,4.185106,2.140000,3.446809,1.080000,0.895307,-1.496504,NaN,NaN,NaN,2024,0.0,0.0,False,False,False,False,home_favored,0.0,San Diego Padres
2577,775328,2024-10-09,Final,Detroit Tigers,Cleveland Guardians,116,114,3.0,0.0,Keider Montero,Alex Cobb,672456,502171,R,R,96.365768,98.757327,11.165049,11.290323,5.083051,3.222449,1.0,-0.125274,1.860602,-2.391559,97.231980,96.107911,1.124069,3.448276,NaN,5.570588,NaN,3.627446,3.098343,3.114558,2.459967,2.751163,3.552830,0.627907,1.528302,-0.876284,0.454487,NaN,NaN,NaN,2024,1.0,0.0,False,False,False,False,home_favored,1.0,Detroit Tigers
2578,775329,2024-10-09,Final,Kansas City Royals,New York Yankees,118,147,2.0,3.0,Seth Lugo,Clarke Schmidt,607625,657376,R,R,99.742087,107.198124,15.909091,17.847025,3.182258,3.510156,0.0,-1.937935,-0.327898,-7.456038,101.587748,109.175214,-7.587467,25.000000,5.000000,1.600000,5.10,3.873019,3.831377,4.297645,3.169300,2.916327,1.887234,0.551020,2.872340,-0.956693,-1.944143,NaN,NaN,NaN,2024,0.0,1.0,False,False,False,False,away_favored,1.0,New York Yankees
2579,775313,2024-10-09,Final,New York Mets,Philadelphia Phillies,121,143,4.0,1.0,Jose Quintana,Ranger Suarez,500779,624133,L,L,103.259086,105.509965,10.041841,16.613419,4.497260,3.299115,1.0,-6.571578,1.198145,-2.250879,108.345466,110.317945,-1.972479,35.000000,0.000000,0.330769,10.60,3.646318,3.355627,3.954869,3.472669,2.885714,3.975000,7.071429,4.500000,-0.760604,0.619373,NaN,NaN,NaN,2024,1.0,0.0,False,False,False,False,home_favored,1.0,New York Mets
2580,775320,2024-10-09,Final,San Diego Padres,Los Angeles Dodgers,135,119,0.0,8.0,Dylan Cease,Ryan Brasier,656302,518489,R,R,104.665885,109.871043,20.866142,18.181818,3.031338,3.242857,0.0,2.684324,-0.211519,-5.205158,107.348602,109.034706,-1.686104,4.347826,20.000000,3.300000,2.10,3.289800,3.636504,3.573273,3.200938,4.642857,2.200000,3.857143,1.350000,1.353057,-1.436504,NaN,NaN,NaN,2024,0.0,2.0,False,True,False,False,away_favored,2.0,Los Angeles Dodgers
2581,775327,2024-10-10,Final,Detroit Tigers,Cleveland Guardians,116,114,4.0,5.0,Reese Olson,Tanner Bibee,681857,676440,R,R,96.365768,98.757327,14.623656,20.140845,3.100000,3.491555,0.0,-5.517189,-0.391555,-2.391559,97.231980,96.107911,1.124069,18.750000,NaN,1.600000,NaN,3.627446,3.098343,3.114558,2.459967,3.322222,4.841935,1.000000,1.741935,-0.305224,1.743592,NaN,NaN,NaN,2024,1.0,0.0,False,False,False,False,home_favored,1.0,Detroit Tigers
2582,775331,2024-10-10,Final,Kansas City Royals,New York Yankees,118,147,1.0,3.0,Michael Wacha,Gerrit Cole,608379,543037,R,R,99.742087,107.198124,14.598540,17.948718,3.586000,3.626316,0.0,-3.350178,-0.040316,-7.456038,101.587748,109.175214,-7.587467,4.545455,17.391304,5.500000,2.05,3.873019,3.831377,4.297645,3.169300,2.856757,2.200000,0.729730,

In [13]:
# V6 model block (kept aligned in spirit with games_today_tomorrow)
s = favorites.copy()

# Stable scaling
SP_KBB_ABS = 3.0
SP_XFIP_ABS = 0.5
OFFENSE_ABS = 10.0
PLATOON_ABS = 10.0
PEN_ABS = 0.75

# Tunable compression / penalty knobs
RISK_W_CONFLICT = 0.30
RISK_W_INSTABILITY = 0.40
RISK_W_SIGNAL_GAP = 0.30
RISK_TANH_ALPHA = 0.80
QUALITY_INSTABILITY_DECAY = 0.90
TRAP_PENALTY_WEIGHT = 0.35

OFF_W_OFFENSE = 0.85
OFF_W_PLATOON = 0.15
PEN_LEG_DAMP = 0.65
COHERENCE_SOFT_MIN = 0.50
COHERENCE_SOFT_RANGE = 0.50

# Missing SP/offense diffs: neutral 0 so norms stay finite
kbb = pd.to_numeric(s["sp_kbb_diff"], errors="coerce").fillna(0.0)
xfi = pd.to_numeric(s["sp_xfip_diff"], errors="coerce").fillna(0.0)
off = pd.to_numeric(s["offense_diff"], errors="coerce").fillna(0.0)
kbb_norm = kbb / SP_KBB_ABS
xfip_norm = -xfi / SP_XFIP_ABS
off_norm = off / OFFENSE_ABS
platoon_norm = s["offense_platoon_diff"].fillna(0) / PLATOON_ABS

# Pen: prefer roll14; neutralize missing values so matrix math is stable
if "home_pen_roll14_fip" in s.columns:
    hr_pen = s["home_pen_roll14_fip"]
    if "home_pen_season_fip" in s.columns:
        hr_pen = hr_pen.combine_first(s["home_pen_season_fip"])
else:
    hr_pen = pd.Series(np.nan, index=s.index)

if "away_pen_roll14_fip" in s.columns:
    ar_pen = s["away_pen_roll14_fip"]
    if "away_pen_season_fip" in s.columns:
        ar_pen = ar_pen.combine_first(s["away_pen_season_fip"])
else:
    ar_pen = pd.Series(np.nan, index=s.index)

pen_gap = ar_pen - hr_pen
pen_norm = (pen_gap / PEN_ABS).fillna(0.0)

sp_vector = (0.65 * kbb_norm) + (0.35 * xfip_norm)
off_vector = (OFF_W_OFFENSE * off_norm) + (OFF_W_PLATOON * platoon_norm)
pen_vector = PEN_LEG_DAMP * pen_norm

signal_matrix = np.vstack([sp_vector, off_vector, pen_vector])
sign_matrix = np.sign(signal_matrix)
mag_matrix = np.abs(signal_matrix)

mean_sign = np.mean(sign_matrix, axis=0)
mean_mag = np.mean(mag_matrix, axis=0)

agreement = 1 - np.nanvar(sign_matrix, axis=0)
direction_purity = np.abs(mean_sign)
mag_consistency = 1 / (1 + np.std(signal_matrix, axis=0))

coherence_score = (0.45 * agreement) + (0.30 * direction_purity) + (0.25 * mag_consistency)
coherence_mult = COHERENCE_SOFT_MIN + COHERENCE_SOFT_RANGE * coherence_score
raw_edge = mean_mag
instability = np.std(signal_matrix, axis=0)

direction_conflict = (
    (np.sign(sp_vector) != np.sign(off_vector)).astype(int)
    + (np.sign(sp_vector) != np.sign(pen_vector)).astype(int)
    + (np.sign(off_vector) != np.sign(pen_vector)).astype(int)
)

risk_penalty_raw = (
    RISK_W_CONFLICT * direction_conflict
    + RISK_W_INSTABILITY * instability
    + RISK_W_SIGNAL_GAP * np.abs(sp_vector - off_vector)
)
risk_penalty = np.tanh(RISK_TANH_ALPHA * risk_penalty_raw)

trap_score = raw_edge * (1 - coherence_score)
quality_score = raw_edge * coherence_mult * np.exp(-QUALITY_INSTABILITY_DECAY * instability) * (1 / (1 + risk_penalty))
risk_adjusted_edge = quality_score - TRAP_PENALTY_WEIGHT * trap_score

prefer_home = sp_vector >= 0
s["favored_team"] = np.where(prefer_home, s["home_team_name"], s["away_team_name"])
s["_mismatch_side"] = np.where(prefer_home, "home_favored", "away_favored")

s["signal_agreement"] = np.sum(sign_matrix > 0, axis=0)
s["signal_conflict"] = np.sum(sign_matrix < 0, axis=0)
s["direction_conflict"] = direction_conflict
s["instability"] = instability
s["coherence_score"] = coherence_score
s["quality_score"] = quality_score
s["risk_adjusted_edge"] = risk_adjusted_edge
s["trap_score"] = trap_score
s["risk_penalty_raw"] = risk_penalty_raw
s["risk_penalty"] = risk_penalty

s["core_matches"] = (
    (np.abs(kbb_norm) > 1).astype(int)
    + (np.abs(xfip_norm) > 1).astype(int)
    + (np.abs(off_norm) > 1).astype(int)
)

# Backtest outcome: did the model's favored side win?
if "home_win" in s.columns:
    hw = pd.to_numeric(s["home_win"], errors="coerce")
    s["favorite_won"] = np.where(prefer_home, hw == 1.0, hw == 0.0)
else:
    s["favorite_won"] = np.nan

# Kalshi is optional in historical backtests.
if {"kalshi_home_implied", "kalshi_away_implied"}.issubset(s.columns):
    s["implied_prob"] = np.where(prefer_home, s["kalshi_home_implied"], s["kalshi_away_implied"])
    s["market_edge"] = s["risk_adjusted_edge"] - s["implied_prob"]
else:
    s["implied_prob"] = np.nan
    s["market_edge"] = np.nan

scored = s.sort_values(["risk_adjusted_edge", "coherence_score"], ascending=[False, False]).reset_index(drop=True)

show_cols = [
    "game_date", "home_team_name", "away_team_name", "favored_team", "_mismatch_side", "favorite_won",
    "risk_adjusted_edge", "quality_score", "coherence_score", "instability",
    "signal_agreement", "signal_conflict", "core_matches",
    "sp_kbb_diff", "sp_xfip_diff", "offense_diff", "offense_platoon_diff",
    "implied_prob", "market_edge", "home_win",
]
show_cols = [c for c in show_cols if c in scored.columns]

print(f"Scored games: {len(scored)}")
display(scored[show_cols].head(25))

# Optional aggregate view (uncomment to use):
# display(
#     scored["favorite_won"]
#     .map({True: "Favorite Won", False: "Favorite Did Not Win"})
#     .fillna("Unknown")
#     .value_counts(dropna=False)
#     .rename_axis("favorite_result")
#     .to_frame("count")
# )

Scored games: 2601


,game_date,home_team_name,away_team_name,favored_team,_mismatch_side,favorite_won,risk_adjusted_edge,quality_score,coherence_score,instability,signal_agreement,signal_conflict,core_matches,sp_kbb_diff,sp_xfip_diff,offense_diff,offense_platoon_diff,implied_prob,market_edge,home_win
0,2024-06-14,Arizona Diamondbacks,Chicago White Sox,Arizona Diamondbacks,home_favored,True,1.013949,1.250400,0.697924,0.160437,3,0,3,6.116612,-1.060454,22.368113,20.795279,NaN,NaN,1.0
1,2024-06-16,Arizona Diamondbacks,Chicago White Sox,Arizona Diamondbacks,home_favored,True,0.862438,1.126438,0.679552,0.268620,3,0,3,5.225564,-1.390528,22.368113,22.107408,NaN,NaN,1.0
2,2024-04-01,Chicago White Sox,Atlanta Braves,Atlanta Braves,away_favored,True,0.813125,0.958271,0.718971,0.057157,0,3,2,-6.064079,0.336895,-14.912075,-12.224252,NaN,NaN,0.0
3,2024-04-22,Minnesota Twins,Chicago White Sox,Minnesota Twins,home_favored,True,0.725316,0.898089,0.694371,0.179894,3,0,2,5.334052,-0.461025,15.193435,14.472390,NaN,NaN,1.0
4,2024-04-06,Pittsburgh Pirates,Baltimore Orioles,Baltimore Orioles,away_favored,False,0.695360,0.803768,0.727755,0.019295,0,3,2,-6.096850,-0.296609,-10.973036,-12.370409,NaN,NaN,1.0
5,2024-04-08,New York Yankees,Miami Marlins,New York Yankees,home_favored,True,0.660343,0.788370,0.709861,0.099514,3,0,2,4.185258,-0.423872,11.817116,11.693984,NaN,NaN,1.0
6,2024-06-06,Chicago White Sox,Boston Red Sox,Boston Red Sox,away_favored,True,0.642654,0.887754,0.668853,0.341450,0,3,3,-4.086846,1.620309,-17.444314,-17.704089,NaN,NaN,0.0
7,2024-07-23,Los Angeles Dodgers,San Francisco Giants,Los Angeles Dodgers,home_favored,True,0.636666,0.768049,0.709635,0.100609,3,0,2,7.645058,0.347077,11.254396,12.645778,NaN,NaN,1.0
8,2024-08-13,Chicago White Sox,New York Yankees,New York Yankees,away_favored,True,0.629166,0.851428,0.670353,0.330742,0,3,3,-7.553829,0.808765,-20.257913,-23.231477,NaN,NaN,0.0
9,2024-05-12,Miami Marlins,Philadelphia Phillies,Philadelphia Phillies,away_favored,False,0.625653,0.734162,0.720467,0.050509,0,3,3,-3.221606,0.688649,-10.128957,-12.383422,NaN,NaN,1.0


In [14]:
# Additional breakdowns by metric bands
band_df = scored.copy()

# Keep known outcomes for cleaner win-rate summaries
band_df = band_df[band_df["favorite_won"].isin([True, False])].copy()

band_specs = {
    "risk_adjusted_edge": [-np.inf, 0.25, 0.50, 0.75, 1.00, np.inf],
    "quality_score": [-np.inf, 0.10, 0.20, 0.30, 0.40, np.inf],
    "coherence_score": [-np.inf, 0.40, 0.50, 0.60, 0.70, np.inf],
    "instability": [-np.inf, 0.80, 1.00, 1.20, 1.50, np.inf],
}

for metric, bins in band_specs.items():
    if metric not in band_df.columns:
        continue

    d = band_df[[metric, "favorite_won"]].dropna().copy()
    if d.empty:
        continue

    d["band"] = pd.cut(d[metric], bins=bins, include_lowest=True)

    out = (
        d.groupby("band", observed=False)["favorite_won"]
        .agg(
            games="count",
            favorite_wins=lambda x: int((x == True).sum()),
            favorite_losses=lambda x: int((x == False).sum()),
            favorite_win_rate="mean",
        )
        .reset_index()
    )
    out["favorite_win_rate"] = out["favorite_win_rate"].round(3)

    print(f"\n{metric} band breakdown")
    display(out)



risk_adjusted_edge band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.25]",2489,1421,1068,0.571
1,"(0.25, 0.5]",88,67,21,0.761
2,"(0.5, 0.75]",21,15,6,0.714
3,"(0.75, 1.0]",2,2,0,1.000
4,"(1.0, inf]",1,1,0,1.000



quality_score band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.1]",1567,852,715,0.544
1,"(0.1, 0.2]",465,277,188,0.596
2,"(0.2, 0.3]",320,200,120,0.625
3,"(0.3, 0.4]",134,88,46,0.657
4,"(0.4, inf]",115,89,26,0.774



coherence_score band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.4]",991,545,446,0.550
1,"(0.4, 0.5]",747,398,349,0.533
2,"(0.5, 0.6]",238,152,86,0.639
3,"(0.6, 0.7]",579,380,199,0.656
4,"(0.7, inf]",46,31,15,0.674



instability band breakdown


,band,games,favorite_wins,favorite_losses,favorite_win_rate
0,"(-inf, 0.8]",853,507,346,0.594
1,"(0.8, 1.0]",345,200,145,0.580
2,"(1.0, 1.2]",303,172,131,0.568
3,"(1.2, 1.5]",318,177,141,0.557
4,"(1.5, inf]",782,450,332,0.575


In [15]:
# Decision / confidence layers + backtest summary
def decision_layer(df: pd.DataFrame) -> pd.Series:
    # Relaxed vs 1.0/0.60/1.2 and 0.55/0.45 — avoids starving BET count
    conditions = [
        (df["risk_adjusted_edge"] > 0.72) & (df["coherence_score"] > 0.52) & (df["instability"] < 1.35),
        (df["risk_adjusted_edge"] > 0.48) & (df["coherence_score"] > 0.40),
    ]
    choices = ["BET", "LEAN"]
    return np.select(conditions, choices, default="PASS")


def confidence_tier(df: pd.DataFrame) -> pd.Series:
    return np.where(
        (df["decision"] == "BET") & (df["coherence_score"] > 0.68),
        "A (Strong Bet)",
        np.where(
            df["decision"] == "BET",
            "B (Playable Bet)",
            np.where(df["decision"] == "LEAN", "C (Lean Only)", "D (Pass)"),
        ),
    )


bt = scored.copy()
bt["decision"] = decision_layer(bt)
bt["tier"] = confidence_tier(bt)

# Evaluate pick correctness only on non-PASS rows
bt_eval = bt[bt["decision"].isin(["BET", "LEAN"])].copy()
bt_eval["pick_home"] = bt_eval["_mismatch_side"].eq("home_favored")
bt_eval["won"] = np.where(bt_eval["pick_home"], bt_eval["home_win"] == 1.0, bt_eval["home_win"] == 0.0)

summary = {
    "rows_scored": int(len(bt)),
    "rows_bet_or_lean": int(len(bt_eval)),
    "win_rate_bet_or_lean": float(bt_eval["won"].mean()) if len(bt_eval) else np.nan,
    "bets_only": int((bt["decision"] == "BET").sum()),
    "leans_only": int((bt["decision"] == "LEAN").sum()),
}

print(summary)
if len(bt_eval):
    by_tier = bt_eval.groupby("tier")["won"].agg(["count", "mean"]).rename(columns={"mean": "win_rate"})
    display(by_tier.sort_values("count", ascending=False))

view_cols = [
    "game_date", "home_team_name", "away_team_name", "favored_team", "decision", "tier",
    "risk_adjusted_edge", "coherence_score", "instability", "home_win", "won",
]
view_cols = [c for c in view_cols if c in bt_eval.columns]
display(bt_eval.sort_values("risk_adjusted_edge", ascending=False)[view_cols].head(50))

# Slate Top-N evaluation (default Top-3 per date) on ALL scored games with known outcomes
TOP_N = 3
if "favorite_won" in scored.columns:
    top_pool = scored[scored["favorite_won"].isin([True, False])].copy()
    top_pool["won"] = top_pool["favorite_won"].astype(bool)
    if len(top_pool):
        bt_topn = (
            top_pool.sort_values(["game_date", "risk_adjusted_edge"], ascending=[True, False])
            .groupby("game_date", as_index=False, group_keys=False)
            .head(TOP_N)
            .copy()
        )
        topn_summary = {
            "top_n": TOP_N,
            "slates": int(bt_topn["game_date"].nunique()),
            "topn_rows": int(len(bt_topn)),
            "topn_win_rate": float(bt_topn["won"].mean()) if len(bt_topn) else np.nan,
        }
        print("Top-N slate summary:", topn_summary)
        by_date = bt_topn.groupby("game_date")["won"].agg(picks="count", wins="sum", win_rate="mean").reset_index()
        display(by_date.head(20))
        top_cols = [c for c in view_cols if c in bt_topn.columns] + [c for c in ["favorite_won"] if c in bt_topn.columns]
        display(bt_topn.sort_values(["game_date", "risk_adjusted_edge"], ascending=[False, False])[top_cols].head(50))

{'rows_scored': 2601, 'rows_bet_or_lean': 21, 'win_rate_bet_or_lean': 0.7142857142857143, 'bets_only': 1, 'leans_only': 20}


,count,win_rate
tier,,
C (Lean Only),20,0.7
B (Playable Bet),1,1.0


,game_date,home_team_name,away_team_name,favored_team,decision,tier,risk_adjusted_edge,coherence_score,instability,home_win,won
0,2024-06-14,Arizona Diamondbacks,Chicago White Sox,Arizona Diamondbacks,BET,B (Playable Bet),1.013949,0.697924,0.160437,1.0,True
1,2024-06-16,Arizona Diamondbacks,Chicago White Sox,Arizona Diamondbacks,LEAN,C (Lean Only),0.862438,0.679552,0.268620,1.0,True
2,2024-04-01,Chicago White Sox,Atlanta Braves,Atlanta Braves,LEAN,C (Lean Only),0.813125,0.718971,0.057157,0.0,True
3,2024-04-22,Minnesota Twins,Chicago White Sox,Minnesota Twins,LEAN,C (Lean Only),0.725316,0.694371,0.179894,1.0,True
4,2024-04-06,Pittsburgh Pirates,Baltimore Orioles,Baltimore Orioles,LEAN,C (Lean Only),0.695360,0.727755,0.019295,1.0,False
5,2024-04-08,New York Yankees,Miami Marlins,New York Yankees,LEAN,C (Lean Only),0.660343,0.709861,0.099514,1.0,True
6,2024-06-06,Chicago White Sox,Boston Red Sox,Boston Red Sox,LEAN,C (Lean Only),0.642654,0.668853,0.341450,0.0,True
7,2024-07-23,Los Angeles Dodgers,San Francisco Giants,Los Angeles Dodgers,LEAN,C (Lean Only),0.636666,0.709635,0.100609,1.0,True
8,2024-08-13,Chicago White Sox,New York Yankees,New York Yankees,LEAN,C (Lean Only),0.629166,0.670353,0.330742,0.0,True
9,2024-05-12,Miami Marlins,Philadelphia Phillies,Philadelphia Phillies,LEAN,C (Lean Only),0.625653,0.720467,0.050509,1.0,False


Top-N slate summary: {'top_n': 3, 'slates': 207, 'topn_rows': 588, 'topn_win_rate': 0.6139455782312925}


,game_date,picks,wins,win_rate
0,2024-03-28,3,1,0.333333
1,2024-03-29,3,1,0.333333
2,2024-03-30,3,2,0.666667
3,2024-03-31,3,2,0.666667
4,2024-04-01,3,2,0.666667
5,2024-04-02,3,3,1.000000
6,2024-04-03,3,3,1.000000
7,2024-04-04,3,3,1.000000
8,2024-04-05,3,3,1.000000
9,2024-04-06,3,1,0.333333


,game_date,home_team_name,away_team_name,favored_team,risk_adjusted_edge,coherence_score,instability,home_win,won,favorite_won
452,2024-10-30,New York Yankees,Los Angeles Dodgers,Los Angeles Dodgers,0.083837,0.639705,0.590156,0.0,True,True
1066,2024-10-29,New York Yankees,Los Angeles Dodgers,Los Angeles Dodgers,-0.063324,0.586995,1.392180,1.0,False,False
1734,2024-10-28,New York Yankees,Los Angeles Dodgers,New York Yankees,-0.188512,0.382193,1.507379,0.0,False,False
560,2024-10-26,Los Angeles Dodgers,New York Yankees,Los Angeles Dodgers,0.039069,0.617619,0.850049,1.0,True,True
451,2024-10-25,Los Angeles Dodgers,New York Yankees,Los Angeles Dodgers,0.083837,0.639705,0.590156,1.0,True,True
307,2024-10-20,Los Angeles Dodgers,New York Mets,Los Angeles Dodgers,0.138192,0.672269,0.317305,1.0,True,True
1122,2024-10-19,Cleveland Guardians,New York Yankees,Cleveland Guardians,-0.070957,0.424078,0.765653,0.0,False,False
942,2024-10-18,New York Mets,Los Angeles Dodgers,Los Angeles Dodgers,-0.042792,0.592600,1.270400,1.0,False,False
1063,2024-10-18,Cleveland Guardians,New York Yankees,Cleveland Guardians,-0.063005,0.420925,0.805871,0.0,False,False
1062,2024-10-17,Cleveland Guardians,New York Yankees,Cleveland Guardians,-0.062991,0.425360,0.749816,1.0,True,True


In [16]:
# Quantile calibration tables + monotonicity checks

cal_df = scored.copy()
if "favorite_won" not in cal_df.columns:
    raise ValueError("favorite_won missing; run V6 scoring cell first.")

cal_df = cal_df[cal_df["favorite_won"].isin([True, False])].copy()
if cal_df.empty:
    raise ValueError("No rows with known favorite_won values for calibration.")

metrics = [
    "risk_adjusted_edge",
    "quality_score",
    "coherence_score",
    "instability",
]


def quantile_calibration_table(df: pd.DataFrame, metric: str, q: int = 10, ascending_good: bool = True):
    d = df[[metric, "favorite_won"]].dropna().copy()
    if d.empty:
        return None, None

    # Handle low-variance metrics safely.
    n_unique = d[metric].nunique(dropna=True)
    q_eff = max(2, min(q, int(n_unique)))

    d["quantile"] = pd.qcut(d[metric], q=q_eff, duplicates="drop")
    tab = (
        d.groupby("quantile", observed=False)["favorite_won"]
        .agg(games="count", favorite_wins="sum", favorite_win_rate="mean")
        .reset_index()
    )
    tab["favorite_losses"] = tab["games"] - tab["favorite_wins"]
    tab["favorite_win_rate"] = tab["favorite_win_rate"].round(3)

    rates = tab["favorite_win_rate"].to_numpy()
    if not ascending_good:
        rates = rates[::-1]
    deltas = np.diff(rates)
    mono_score = float((deltas >= 0).mean()) if len(deltas) else np.nan
    is_monotonic = bool((deltas >= 0).all()) if len(deltas) else True

    summary = {
        "metric": metric,
        "quantile_bins": int(len(tab)),
        "rows_used": int(tab["games"].sum()),
        "win_rate_first_bin": float(tab.iloc[0]["favorite_win_rate"]),
        "win_rate_last_bin": float(tab.iloc[-1]["favorite_win_rate"]),
        "lift_last_minus_first": float(tab.iloc[-1]["favorite_win_rate"] - tab.iloc[0]["favorite_win_rate"]),
        "is_monotonic_expected_direction": is_monotonic,
        "monotonicity_fraction": round(mono_score, 3) if not np.isnan(mono_score) else np.nan,
    }
    return tab, summary


# For instability, lower values are generally better; reverse expected direction.
expected_ascending = {
    "risk_adjusted_edge": True,
    "quality_score": True,
    "coherence_score": True,
    "instability": False,
}

summaries = []
for m in metrics:
    if m not in cal_df.columns:
        continue
    tab, s = quantile_calibration_table(cal_df, m, q=10, ascending_good=expected_ascending[m])
    if tab is None:
        continue
    print(f"\n{m}: quantile calibration")
    display(tab)
    summaries.append(s)

if summaries:
    print("\nMonotonicity summary")
    display(pd.DataFrame(summaries).sort_values("metric").reset_index(drop=True))


risk_adjusted_edge: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(-3.237, -0.392]",261,157,0.602,104
1,"(-0.392, -0.281]",260,140,0.538,120
2,"(-0.281, -0.211]",260,148,0.569,112
3,"(-0.211, -0.153]",260,145,0.558,115
4,"(-0.153, -0.104]",260,145,0.558,115
5,"(-0.104, -0.0592]",260,142,0.546,118
6,"(-0.0592, -0.0175]",260,140,0.538,120
7,"(-0.0175, 0.0571]",260,143,0.550,117
8,"(0.0571, 0.157]",260,169,0.650,91
9,"(0.157, 1.014]",260,177,0.681,83



quality_score: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(-0.000999154, 0.0481]",261,155,0.594,106
1,"(0.0481, 0.0622]",260,136,0.523,124
2,"(0.0622, 0.0715]",260,127,0.488,133
3,"(0.0715, 0.0789]",260,143,0.550,117
4,"(0.0789, 0.0872]",260,136,0.523,124
5,"(0.0872, 0.0997]",260,154,0.592,106
6,"(0.0997, 0.124]",260,143,0.550,117
7,"(0.124, 0.213]",260,166,0.638,94
8,"(0.213, 0.296]",260,162,0.623,98
9,"(0.296, 1.25]",260,184,0.708,76



coherence_score: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(0.296, 0.361]",261,150,0.575,111
1,"(0.361, 0.376]",260,132,0.508,128
2,"(0.376, 0.388]",260,149,0.573,111
3,"(0.388, 0.402]",260,145,0.558,115
4,"(0.402, 0.418]",260,139,0.535,121
5,"(0.418, 0.441]",260,140,0.538,120
6,"(0.441, 0.574]",260,145,0.558,115
7,"(0.574, 0.614]",260,162,0.623,98
8,"(0.614, 0.651]",260,176,0.677,84
9,"(0.651, 0.728]",260,168,0.646,92



instability: quantile calibration


,quantile,games,favorite_wins,favorite_win_rate,favorite_losses
0,"(0.0183, 0.407]",261,161,0.617,100
1,"(0.407, 0.579]",260,146,0.562,114
2,"(0.579, 0.752]",260,154,0.592,106
3,"(0.752, 0.92]",260,155,0.596,105
4,"(0.92, 1.066]",260,145,0.558,115
5,"(1.066, 1.245]",260,152,0.585,108
6,"(1.245, 1.501]",260,144,0.554,116
7,"(1.501, 1.76]",260,151,0.581,109
8,"(1.76, 2.242]",260,140,0.538,120
9,"(2.242, 16.28]",260,158,0.608,102



Monotonicity summary


,metric,quantile_bins,rows_used,win_rate_first_bin,win_rate_last_bin,lift_last_minus_first,is_monotonic_expected_direction,monotonicity_fraction
0,coherence_score,10,2601,0.575,0.646,0.071,False,0.556
1,instability,10,2601,0.617,0.608,-0.009,False,0.444
2,quality_score,10,2601,0.594,0.708,0.114,False,0.444
3,risk_adjusted_edge,10,2601,0.602,0.681,0.079,False,0.556


In [17]:
# Post-model calibration (logistic + isotonic), quantile diagnostics, and YoY-style summary schema
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

work = scored.copy()
work = work[work["favorite_won"].isin([True, False])].copy()
if work.empty:
    raise ValueError("No rows with favorite_won for calibration.")

work["y"] = work["favorite_won"].astype(int)
work["score_raw"] = pd.to_numeric(work["risk_adjusted_edge"], errors="coerce")
work = work.dropna(subset=["score_raw", "y"]).copy()

# Time-aware split by game_date to reduce leakage.
unique_dates = sorted(pd.to_datetime(work["game_date"]).dropna().unique())
if len(unique_dates) < 5:
    raise ValueError("Not enough unique dates for calibration split.")
cut_idx = max(1, int(len(unique_dates) * 0.7))
train_dates = set(unique_dates[:cut_idx])
valid_dates = set(unique_dates[cut_idx:])

train = work[work["game_date"].isin(train_dates)].copy()
valid = work[work["game_date"].isin(valid_dates)].copy()
if train.empty or valid.empty:
    raise ValueError("Train/valid split empty; adjust date window.")

xtr = train[["score_raw"]].to_numpy()
ytr = train["y"].to_numpy()
xva = valid[["score_raw"]].to_numpy()
yva = valid["y"].to_numpy()

# Baseline probability proxy from score scaling
smin, smax = float(train["score_raw"].min()), float(train["score_raw"].max())
den = (smax - smin) if (smax - smin) > 1e-12 else 1.0
train["p_raw"] = ((train["score_raw"] - smin) / den).clip(0, 1)
valid["p_raw"] = ((valid["score_raw"] - smin) / den).clip(0, 1)

# Logistic (Platt-style)
logit = LogisticRegression(max_iter=2000)
logit.fit(xtr, ytr)
train["p_model_logit"] = logit.predict_proba(xtr)[:, 1]
valid["p_model_logit"] = logit.predict_proba(xva)[:, 1]

# Isotonic on raw score
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(train["score_raw"].to_numpy(), ytr)
train["p_model_iso"] = iso.predict(train["score_raw"].to_numpy())
valid["p_model_iso"] = iso.predict(valid["score_raw"].to_numpy())

# Write calibrated probabilities back to scored
scored = scored.copy()
for col in ["p_model_logit", "p_model_iso"]:
    scored[col] = np.nan
scored.loc[train.index, "p_model_logit"] = train["p_model_logit"]
scored.loc[valid.index, "p_model_logit"] = valid["p_model_logit"]
scored.loc[train.index, "p_model_iso"] = train["p_model_iso"]
scored.loc[valid.index, "p_model_iso"] = valid["p_model_iso"]


def metric_row(name: str, y_true: np.ndarray, p: np.ndarray) -> dict:
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return {
        "model": name,
        "brier": float(brier_score_loss(y_true, p)),
        "logloss": float(log_loss(y_true, p)),
        "auc": float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) > 1 else np.nan,
    }

metric_rows = [
    metric_row("raw_scaled", yva, valid["p_raw"].to_numpy()),
    metric_row("logit", yva, valid["p_model_logit"].to_numpy()),
    metric_row("isotonic", yva, valid["p_model_iso"].to_numpy()),
]
metrics_df = pd.DataFrame(metric_rows).sort_values("brier")
print("Validation calibration metrics")
display(metrics_df)


def quantile_hit_rate(df: pd.DataFrame, score_col: str, y_col: str = "y", q: int = 10):
    d = df[[score_col, y_col]].dropna().copy()
    if d.empty or d[score_col].nunique() < 2:
        return pd.DataFrame(), {"metric": score_col, "monotonicity_fraction": np.nan, "lift_last_minus_first": np.nan}
    q_eff = min(q, int(d[score_col].nunique()))
    d["quantile"] = pd.qcut(d[score_col], q=q_eff, duplicates="drop")
    tab = d.groupby("quantile", observed=False)[y_col].agg(games="count", win_rate="mean").reset_index()
    tab["win_rate"] = tab["win_rate"].round(3)
    deltas = np.diff(tab["win_rate"].to_numpy())
    mono = float((deltas >= 0).mean()) if len(deltas) else np.nan
    lift = float(tab.iloc[-1]["win_rate"] - tab.iloc[0]["win_rate"]) if len(tab) else np.nan
    return tab, {"metric": score_col, "monotonicity_fraction": round(mono, 3) if not np.isnan(mono) else np.nan, "lift_last_minus_first": lift}

quant_targets = ["score_raw", "quality_score", "coherence_score", "p_model_logit", "p_model_iso"]
mono_rows = []
for col in quant_targets:
    if col not in valid.columns:
        continue
    qt, ms = quantile_hit_rate(valid, col, y_col="y", q=10)
    if len(qt):
        print(f"\nValidation quantile hit-rate: {col}")
        display(qt)
    mono_rows.append(ms)

mono_df = pd.DataFrame(mono_rows)
print("\nMonotonicity summary")
display(mono_df)

# Standardized summary schema for cross-year comparison
summary_row = {
    "season": int(SEASON),
    "rows_scored": int(len(scored)),
    "rows_with_outcome": int(len(work)),
    "train_rows": int(len(train)),
    "valid_rows": int(len(valid)),
    "top_n_default": 3,
}

# Include top-N result if prior cell produced it
if "bt_topn" in globals() and isinstance(bt_topn, pd.DataFrame) and len(bt_topn):
    summary_row["topn_rows"] = int(len(bt_topn))
    summary_row["topn_win_rate"] = float(bt_topn["won"].mean())
else:
    summary_row["topn_rows"] = np.nan
    summary_row["topn_win_rate"] = np.nan

best_cal = metrics_df.iloc[0]
summary_row["best_model_by_brier"] = best_cal["model"]
summary_row["best_brier"] = float(best_cal["brier"])
summary_row["best_logloss"] = float(best_cal["logloss"])
summary_row["best_auc"] = float(best_cal["auc"]) if pd.notna(best_cal["auc"]) else np.nan

year_summary = pd.DataFrame([summary_row])
print("\nYear summary row")
display(year_summary)

Validation calibration metrics


,model,brier,logloss,auc
1,logit,0.244682,0.682492,0.479062
2,isotonic,0.245381,0.684665,0.478343
0,raw_scaled,0.271020,0.747720,0.479062



Validation quantile hit-rate: score_raw


,quantile,games,win_rate
0,"(-1.115, -0.329]",60,0.617
1,"(-0.329, -0.24]",60,0.617
2,"(-0.24, -0.188]",60,0.583
3,"(-0.188, -0.137]",59,0.627
4,"(-0.137, -0.0951]",60,0.567
5,"(-0.0951, -0.0585]",60,0.633
6,"(-0.0585, -0.0173]",59,0.475
7,"(-0.0173, 0.0736]",60,0.483
8,"(0.0736, 0.158]",60,0.683
9,"(0.158, 0.574]",60,0.550



Validation quantile hit-rate: quality_score


...

,quantile,games,win_rate
0,"(0.5404, 0.5478]",243,0.617
1,"(0.5478, 0.5479]",160,0.556
2,"(0.5479, 0.5826]",64,0.422
3,"(0.5826, 0.625]",18,0.889
4,"(0.625, 0.6404]",84,0.595
5,"(0.6404, 0.8485]",29,0.586



Monotonicity summary


,metric,monotonicity_fraction,lift_last_minus_first
0,score_raw,0.556,-0.067
1,quality_score,0.444,-0.017
2,coherence_score,0.556,-0.083
3,p_model_logit,0.556,-0.067
4,p_model_iso,0.200,-0.031



Year summary row


,season,rows_scored,rows_with_outcome,train_rows,valid_rows,top_n_default,topn_rows,topn_win_rate,best_model_by_brier,best_brier,best_logloss,best_auc
0,2024,2601,2601,2003,598,3,588,0.613946,logit,0.244682,0.682492,0.479062


## Metric definitions and acceptance bars (winner prediction)

- **Brier score**: mean squared error of predicted probability vs actual outcome (`0/1`). Lower is better.
- **Log loss**: penalizes confident wrong probabilities more than Brier. Lower is better.
- **AUC**: probability that a random win is ranked above a random loss. `0.5` is random, `1.0` is perfect.
- **Monotonicity fraction**: fraction of adjacent quantile bins where win rate moves in expected direction.
- **Lift (last-first)**: difference in win rate between highest and lowest quantile bins.
- **Top-N win rate**: win rate among the top-N ranked picks per slate date.

Suggested minimum bars for practical actionability (outright winners):

- `AUC >= 0.53`
- `best_brier <= 0.245`
- `monotonicity_fraction >= 0.60` on calibrated probability
- `lift_last_minus_first >= 0.03` on calibrated probability
- `topn_win_rate >= 0.54` for `TOP_N=3`


In [18]:
# Acceptance-bar check for actionability (winner prediction)
# Requires year_summary, metrics_df, mono_df from the calibration cell.

if "year_summary" not in globals() or "metrics_df" not in globals() or "mono_df" not in globals():
    raise ValueError("Run calibration/summary cell first.")

bars = {
    "min_auc": 0.53,
    "max_brier": 0.245,
    "min_mono": 0.60,
    "min_lift": 0.03,
    "min_topn_win_rate": 0.54,
}

ys = year_summary.iloc[0]
prob_row = mono_df[mono_df["metric"].isin(["p_model_logit", "p_model_iso"])].copy()
if prob_row.empty:
    mono_best = np.nan
    lift_best = np.nan
else:
    mono_best = float(prob_row["monotonicity_fraction"].max())
    lift_best = float(prob_row["lift_last_minus_first"].max())

checks = pd.DataFrame(
    [
        {"check": "AUC", "value": float(ys.get("best_auc", np.nan)), "bar": bars["min_auc"], "pass": float(ys.get("best_auc", np.nan)) >= bars["min_auc"]},
        {"check": "Brier", "value": float(ys.get("best_brier", np.nan)), "bar": bars["max_brier"], "pass": float(ys.get("best_brier", np.nan)) <= bars["max_brier"]},
        {"check": "Monotonicity (prob)", "value": mono_best, "bar": bars["min_mono"], "pass": mono_best >= bars["min_mono"] if pd.notna(mono_best) else False},
        {"check": "Lift (prob)", "value": lift_best, "bar": bars["min_lift"], "pass": lift_best >= bars["min_lift"] if pd.notna(lift_best) else False},
        {"check": "Top-3 win rate", "value": float(ys.get("topn_win_rate", np.nan)), "bar": bars["min_topn_win_rate"], "pass": float(ys.get("topn_win_rate", np.nan)) >= bars["min_topn_win_rate"]},
    ]
)

print("Actionability checks")
display(checks)
print("Passed", int(checks["pass"].sum()), "of", len(checks), "checks")

Actionability checks


,check,value,bar,pass
0,AUC,0.479062,0.530,False
1,Brier,0.244682,0.245,True
2,Monotonicity (prob),0.556000,0.600,False
3,Lift (prob),-0.031000,0.030,False
4,Top-3 win rate,0.613946,0.540,True


Passed 2 of 5 checks
